# Módulo 2 · Clase 2 — Git local avanzado

**Curso de Inteligencia Artificial — Nivel básico**

---

## ¿Qué vamos a ver hoy?

| # | Bloque | Qué aprenderás |
|---|--------|----------------|
| 1 | `git diff` | Ver exactamente qué cambió y dónde |
| 2 | Deshacer cambios | `restore`, `revert`, `reset` — cuándo usar cada uno |
| 3 | `git stash` | Guardar trabajo a medias sin commitear |
| 4 | `git commit --amend` | Corregir el último commit |
| 5 | `git log` avanzado | Filtrar y formatear el historial |
| 6 | `git tag` | Marcar versiones importantes |
| 7 | Práctica | Refactorizar un proyecto con historial limpio |

> 💡 **Idea central:** el 80% de los problemas que tiene la gente con Git no son de colaboración — son de «cómo deshago esto». Esta clase cubre exactamente eso, con la distinción clave entre operaciones **seguras** (que no reescriben historial) y **peligrosas** (que sí lo hacen).

In [ ]:
# Preparar el entorno de demo — creamos un repo con historial existente
import os, shutil
from pathlib import Path

REPO = Path.home() / "git_avanzado_demo"
if REPO.exists():
    shutil.rmtree(REPO)
REPO.mkdir()
os.chdir(REPO)

# Inicializar y configurar
!git init -q
!git config user.name  "Estudiante Demo"
!git config user.email "demo@curso.com"

# Crear historial inicial
Path(".gitignore").write_text("__pycache__/\n*.pyc\n.env\nvenv/\n")
Path("app.py").write_text("""
def saludar(nombre):
    return f'Hola, {nombre}'

def despedir(nombre):
    return f'Adiós, {nombre}'
""")
Path("utils.py").write_text("""
def mayusculas(texto):
    return texto.upper()
""")

!git add -A && git commit -q -m "init: estructura inicial"

# Segundo commit
with open("app.py", "a") as f:
    f.write("\ndef calcular(a, b):\n    return a + b\n")
!git add app.py && git commit -q -m "feat: añadir función calcular"

# Tercer commit
Path("config.py").write_text("DEBUG = True\nVERSION = '1.0'\n")
!git add config.py && git commit -q -m "chore: añadir configuración"

print(f"✅ Repo demo listo en {REPO}")
!git log --oneline

---
## 1. `git diff` — Ver exactamente qué cambió

`git diff` muestra las diferencias entre distintos estados del repositorio. Es tu herramienta de revisión antes de commitear.

```
git diff                   →  working directory vs staging area
git diff --staged          →  staging area vs último commit
git diff HEAD              →  working directory vs último commit
git diff <hash1> <hash2>   →  entre dos commits
git diff main..feature     →  entre dos ramas
```

**Leer la salida de `git diff`:**
```diff
--- a/app.py        ← versión anterior
+++ b/app.py        ← versión nueva
@@ -3,4 +3,6 @@    ← líneas 3-4 antes, 3-6 después
 def saludar(nombre):      ← contexto (sin cambio)
-    return f'Hola'        ← línea eliminada (rojo)
+    return f'Hola, {nombre}'  ← línea añadida (verde)
```

In [ ]:
# Hacer un cambio sin añadir al stage
with open("app.py", "a") as f:
    f.write("\ndef nueva_funcion():\n    pass\n")

print("=== git diff (working vs stage) ===")
!git diff app.py

In [ ]:
!git add app.py

print("=== git diff --staged (stage vs último commit) ===")
!git diff --staged app.py

In [ ]:
# Limpiar el stage para los siguientes ejemplos
!git restore --staged app.py
!git restore app.py

print("=== Diff entre primer y tercer commit ===")
!git diff HEAD~2 HEAD

In [ ]:
# Ver solo el resumen (qué archivos y cuántas líneas)
print("=== git diff --stat ===")
!git diff HEAD~2 HEAD --stat

print("\n=== git diff --name-only ===")
!git diff HEAD~2 HEAD --name-only

---
## 2. Deshacer cambios — `restore`, `revert`, `reset`

Esta es la parte más importante de la clase. Hay tres comandos para deshacer cosas y **no son intercambiables**.

```
¿QUÉ QUIERES DESHACER?          COMANDO
─────────────────────────────────────────────────────────
Cambios en el working directory  git restore <archivo>
Archivo añadido al stage         git restore --staged <archivo>
Un commit ya hecho (de forma     git revert <hash>        ← SEGURO
segura, sin borrar historial)
Volver atrás moviendo HEAD       git reset --soft  HEAD~n  ← deshacer commit, guardar stage
(con cuidado)                    git reset --mixed HEAD~n  ← deshacer commit, guardar working
                                 git reset --hard  HEAD~n  ← borrar TODO ← DESTRUCTIVO
```

### La regla de oro

> **Nunca uses `reset` sobre commits que ya has subido a un repositorio remoto compartido.**  
> Usa siempre `revert` para deshacer en historial público.

In [ ]:
# ── git restore: deshacer cambios en el working directory ─
print("ANTES:")
print(Path("utils.py").read_text())

# Modificar el archivo
Path("utils.py").write_text("# ARCHIVO ROTO\ndef mayusculas(texto):\n    return 'ERROR'\n")
print("\nDESPUÉS de modificar:")
print(Path("utils.py").read_text())

# Deshacer — vuelve al estado del último commit
!git restore utils.py
print("TRAS git restore:")
print(Path("utils.py").read_text())

In [ ]:
# ── git restore --staged: sacar del stage sin perder cambios ─
Path("utils.py").write_text("# Cambio importante\ndef mayusculas(texto):\n    return texto.upper()\n")
!git add utils.py

print("Estado tras git add:")
!git status

# Sacar del stage — el cambio sigue en el archivo
!git restore --staged utils.py
print("\nEstado tras git restore --staged:")
!git status
print("\nEl archivo aún tiene el cambio:")
print(Path("utils.py").read_text())

# Limpiar para el siguiente ejemplo
!git restore utils.py

In [ ]:
# ── git revert: deshacer un commit de forma SEGURA ────────
# Crea un nuevo commit que invierte los cambios del anterior
# El historial original se preserva — SIEMPRE es seguro

print("Historial antes del revert:")
!git log --oneline

# Revertir el último commit (el de config.py)
!git revert HEAD --no-edit

print("\nHistorial DESPUÉS del revert:")
!git log --oneline
print("\nconfig.py ya no existe:")
!ls

In [ ]:
# ── git reset --soft: mover HEAD sin tocar los archivos ───
# Ideal para: "quiero unir mis últimos N commits en uno solo"

# Primero re-creamos el commit revertido para el ejemplo
Path("config.py").write_text("DEBUG = True\nVERSION = '1.0'\n")
Path("extra.py").write_text("# Extra\n")
!git add config.py && git commit -q -m "chore: config"
!git add extra.py  && git commit -q -m "chore: extra"

print("Historial con dos commits pequeños:")
!git log --oneline

# Deshacer los 2 últimos commits pero mantener los archivos en el stage
!git reset --soft HEAD~2

print("\nTras reset --soft HEAD~2:")
!git log --oneline
print("\ngit status — los archivos siguen en el stage:")
!git status

# Ahora podemos hacer un solo commit limpio
!git commit -q -m "chore: añadir configuración y módulo extra"
print("\nHistorial limpio:")
!git log --oneline

In [ ]:
# ── Comparación de los tres tipos de reset ────────────────
comparacion = """
╔══════════════╦══════════════╦══════════════╦══════════════╗
║ Comando      ║ Mueve HEAD   ║ Staging area ║ Working dir  ║
╠══════════════╬══════════════╬══════════════╬══════════════╣
║ --soft       ║ Sí           ║ Intacta      ║ Intacto      ║
║ --mixed      ║ Sí           ║ Limpia       ║ Intacto      ║
║ --hard       ║ Sí           ║ Limpia       ║ LIMPIA ⚠️    ║
╚══════════════╩══════════════╩══════════════╩══════════════╝

--soft  → deshacer commit, mantener TODO              (lo más suave)
--mixed → deshacer commit y stage, mantener archivos  (el defecto)
--hard  → deshacer TODO, incluyendo cambios en disco  (irreversible)
"""
print(comparacion)

---
## 3. `git stash` — Guardar trabajo a medias

Escenario clásico: estás trabajando en una función a medias y de repente tienes que cambiar de rama para arreglar un bug urgente. No quieres commitear código incompleto, pero tampoco perderlo.

**`git stash`** guarda tus cambios en una pila temporal y deja el working directory limpio.

```bash
git stash              # guardar cambios (push implícito)
git stash push -m "descripción"   # con nombre descriptivo
git stash list         # ver todos los stashes guardados
git stash pop          # recuperar el más reciente y eliminarlo
git stash apply        # recuperar sin eliminar del stash
git stash drop         # eliminar el stash más reciente
git stash clear        # eliminar todos los stashes
```

In [ ]:
# Simular trabajo a medias
with open("app.py", "a") as f:
    f.write("\n# TODO: esta función está a medias\ndef funcion_incompleta():\n    # ..."
            "\n    pass\n")
Path("borrador.py").write_text("# borrador temporal\n")

print("Estado con trabajo a medias:")
!git status

In [ ]:
# Guardar en el stash con descripción
!git add -A
!git stash push -m "WIP: función de exportación a medias"

print("Estado tras stash — working directory limpio:")
!git status

print("\nStashes guardados:")
!git stash list

In [ ]:
# Simular que arreglamos el bug urgente
with open("utils.py", "a") as f:
    f.write("\ndef minusculas(texto):\n    return texto.lower()\n")
!git add utils.py && git commit -q -m "fix: añadir función minusculas"

print("Bug arreglado. Recuperamos el trabajo del stash:")
!git stash pop

print("\nEstado tras pop — el trabajo vuelve:")
!git status

# Limpiar
!git restore --staged . && git restore .
!git clean -fd

---
## 4. `git commit --amend` — Corregir el último commit

`--amend` reemplaza el último commit con uno nuevo. Úsalo para:
- Corregir un typo en el mensaje.
- Añadir un archivo que olvidaste incluir.
- Hacer un pequeño cambio que debería ser parte del mismo commit.

> ⚠️ **Solo en commits que NO has subido al remoto.** `--amend` reescribe el historial — si ya lo subiste, causará problemas a tus compañeros.

In [ ]:
# Commit con mensaje erróneo
Path("nuevo_modulo.py").write_text("# Módulo nuevo\n")
!git add nuevo_modulo.py
!git commit -m "feat: nuev modulo"   # typo en el mensaje

print("Historial con mensaje erróneo:")
!git log --oneline

In [ ]:
# Corregir solo el mensaje
!git commit --amend -m "feat: añadir nuevo módulo"

print("Historial tras amend:")
!git log --oneline

In [ ]:
# Añadir un archivo que olvidamos incluir en el commit anterior
Path("nuevo_modulo_test.py").write_text("# Tests del módulo nuevo\n")
!git add nuevo_modulo_test.py

# --no-edit mantiene el mensaje sin abrir el editor
!git commit --amend --no-edit

print("Ahora el commit incluye los dos archivos:")
!git show --stat HEAD

---
## 5. `git log` avanzado — Buscar en el historial

`git log` tiene docenas de opciones. Estas son las más útiles en el trabajo diario.

In [ ]:
# Crear más commits para tener historial rico
for i in range(3):
    Path(f"archivo_{i}.py").write_text(f"# Archivo {i}\n")
    !git add . && git commit -q -m f"feat: archivo {i}"

print("=== Historial completo ===")
!git log --oneline

In [ ]:
# Filtrar por mensaje
print("=== Solo commits con 'feat' ===")
!git log --oneline --grep="feat"

print("\n=== Solo commits con 'chore' ===")
!git log --oneline --grep="chore"

In [ ]:
# -S busca commits que añadieron o quitaron una cadena de texto
# Útil para: ¿en qué commit se introdujo esta función?
print("=== Commits que tocaron la cadena 'calcular' ===")
!git log --oneline -S "calcular"

In [ ]:
# Formato personalizado
print("=== Formato personalizado ===")
# %h=hash corto, %ad=fecha, %s=asunto, %an=autor
!git log --pretty=format:"%h  %ad  %an  %s" --date=short

print("\n=== Con gráfico de ramas ===")
!git log --oneline --graph --all

print("\n=== Últimos 3 commits ===")
!git log --oneline -3

In [ ]:
# Ver historial de un archivo concreto
print("=== Historial de app.py ===")
!git log --oneline -- app.py

print("\n=== Qué cambió en app.py en cada commit ===")
!git log -p -- app.py

In [ ]:
# git blame: quién escribió cada línea y en qué commit
# Útil para entender por qué algo está escrito de cierta manera
print("=== git blame app.py ===")
!git blame app.py

---
## 6. `git tag` — Marcar versiones

Los tags son etiquetas permanentes que apuntan a un commit concreto. Se usan para marcar versiones de software: `v1.0`, `v2.3.1`, etc.

| Tipo | Comando | Cuándo |
|------|---------|--------|
| **Lightweight** | `git tag v1.0` | Solo una etiqueta, sin metadatos |
| **Annotated** | `git tag -a v1.0 -m "Versión 1.0"` | Con autor, fecha y mensaje — el recomendado |

In [ ]:
# Crear un tag anotado en el commit actual
!git tag -a v1.0 -m "Primera versión estable"

# Listar tags
print("=== Tags del repositorio ===")
!git tag

# Ver los detalles del tag
print("\n=== Detalles del tag v1.0 ===")
!git show v1.0 --stat

# En el historial aparece el tag
print("\n=== Historial con tags ===")
!git log --oneline --decorate -5

In [ ]:
# Etiquetar un commit anterior
HASH_SEGUNDO = !git log --oneline
hash_segundo = HASH_SEGUNDO[-2].split()[0]  # el segundo commit

!git tag -a v0.1 {hash_segundo} -m "Versión inicial de prueba"

print("Tags en orden de versión:")
!git tag -l --sort=version:refname

print("\nHistorial con todos los tags:")
!git log --oneline --decorate

---
## 7. Práctica — Refactorizar con historial limpio

El objetivo de esta práctica es aplicar todos los comandos de la clase en un escenario realista: recibes un proyecto con historial caótico y tienes que limpiarlo y mejorarlo.

---

### Ejercicio 1 — Exploración y diagnóstico

**Enunciado:** Usando el repo de demo que ya tienes, responde estas preguntas **solo con comandos Git** (sin abrir los archivos):

1. ¿Cuántos commits tiene el repositorio?
2. ¿En qué commit se introdujo la función `calcular`?
3. ¿Cuáles son los archivos trackeados actualmente?
4. ¿Qué cambió entre el primer y el último commit? (resumen de archivos)
5. ¿Quién es el autor de cada línea de `app.py`?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 1</summary>

```python
# 1. Número de commits
!git rev-list --count HEAD

# 2. Commit donde se introdujo 'calcular'
!git log --oneline -S "calcular"

# 3. Archivos trackeados
!git ls-files

# 4. Resumen de cambios entre primer y último commit
!git diff HEAD~$(git rev-list --count HEAD | tr -d '\n')..HEAD --stat --diff-filter=A
# Forma más legible:
PRIMERO = !git log --oneline | tail -1
hash_primero = PRIMERO[0].split()[0]
!git diff {hash_primero} HEAD --name-status

# 5. Autor de cada línea de app.py
!git blame app.py
```

**Nota:** `git log -S "calcular"` es el «pickaxe» — busca commits que añadieron o eliminaron esa cadena exacta. Imprescindible para investigar cuándo apareció (o desapareció) algo en el código.
</details>

---
### Ejercicio 2 — Revertir sin perder historial

**Enunciado:**
1. Crea un archivo `error.py` con contenido erróneo y haz un commit: `"feat: módulo de errores"`.
2. Verifica que aparece en el historial.
3. Usa `git revert` para deshacer ese commit de forma segura.
4. Verifica que el historial tiene ahora **cuatro commits** — el original más el revert — y que `error.py` ya no existe.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 2</summary>

```python
# 1. Crear el commit erróneo
Path("error.py").write_text("# Este módulo no debería existir\n")
!git add error.py
!git commit -m "feat: módulo de errores"

# 2. Verificar en el historial
print("Historial con el commit erróneo:")
!git log --oneline

# 3. Revertir de forma segura
!git revert HEAD --no-edit

# 4. Verificar
print("\nHistorial tras revert (dos commits nuevos, historial intacto):")
!git log --oneline
print("\n¿Existe error.py?")
!ls error.py 2>/dev/null || echo "No existe — correcto"
```

**Por qué `revert` y no `reset`:** `revert` añade un commit nuevo que invierte los cambios. El commit erróneo sigue en el historial — esto es bueno porque cualquier persona que haya clonado el repo verá exactamente qué pasó. `reset --hard` borraría el commit del historial, lo que causaría problemas si alguien ya lo tiene descargado.
</details>

---
### Ejercicio 3 — Limpiar commits con `reset --soft`

**Enunciado:** Simula el escenario de trabajar desordenado y luego limpiar:
1. Haz tres commits pequeños y mal nombrados seguidos:
   - `"wip"` → crea `modulo_a.py`
   - `"mas cambios"` → crea `modulo_b.py`
   - `"casi"` → crea `modulo_c.py`
2. Usa `git reset --soft HEAD~3` para unir los tres en uno solo.
3. Haz un commit limpio: `"feat: añadir módulos a, b y c"`.
4. Verifica el historial.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 3</summary>

```python
# 1. Tres commits desordenados
Path("modulo_a.py").write_text("# A\n")
!git add modulo_a.py && git commit -q -m "wip"

Path("modulo_b.py").write_text("# B\n")
!git add modulo_b.py && git commit -q -m "mas cambios"

Path("modulo_c.py").write_text("# C\n")
!git add modulo_c.py && git commit -q -m "casi"

print("Historial desordenado:")
!git log --oneline

# 2. Unir los tres commits
!git reset --soft HEAD~3

print("\nTras reset --soft — los archivos siguen en el stage:")
!git status

# 3. Un commit limpio
!git commit -m "feat: añadir módulos a, b y c"

# 4. Historial limpio
print("\nHistorial limpio:")
!git log --oneline
print("\nEl commit contiene los tres archivos:")
!git show --stat HEAD
```

**Este patrón** — trabajar rápido con commits WIP y luego limpiar con `reset --soft` antes de subir — es muy común en el trabajo profesional. Nadie en el equipo ve el historial sucio.
</details>

---
### Ejercicio 4 — El flujo completo con stash

**Enunciado:** Simula el escenario de interrupción más común en el trabajo diario:

1. Empieza a escribir una función nueva en `app.py` (sin commitear).
2. «Te llega un bug urgente»: guarda tu trabajo con `git stash push -m "WIP: función de exportación"`.
3. Arregla el bug: añade una línea a `utils.py` y commitea con `"fix: corregir función de limpieza"`.
4. Recupera tu trabajo del stash con `git stash pop`.
5. Termina la función y haz el commit final.
6. Verifica el historial completo.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 4</summary>

```python
# 1. Trabajo a medias
with open("app.py", "a") as f:
    f.write("\n# TODO: terminar\ndef exportar_csv():\n    pass\n")
print("Estado con trabajo a medias:")
!git status

# 2. Guardar en stash
!git add app.py
!git stash push -m "WIP: función de exportación"
print("\nTras stash — working directory limpio:")
!git status

# 3. Arreglar el bug urgente
with open("utils.py", "a") as f:
    f.write("\ndef quitar_espacios(texto):\n    return texto.strip()\n")
!git add utils.py
!git commit -m "fix: corregir función de limpieza"

# 4. Recuperar el trabajo del stash
!git stash pop
print("\nTras stash pop — el trabajo vuelve:")
!git status

# 5. Terminar la función y commitear
contenido = Path("app.py").read_text()
contenido = contenido.replace(
    "# TODO: terminar\ndef exportar_csv():\n    pass",
    "def exportar_csv(datos, ruta):\n    with open(ruta, 'w') as f:\n        f.write(','.join(datos))"
)
Path("app.py").write_text(contenido)
!git add app.py
!git commit -m "feat: implementar exportación a CSV"

# 6. Historial final
print("\nHistorial completo:")
!git log --oneline
```

**Observa el historial:** el commit del bug (`fix`) aparece entre los dos de la nueva función (`stash` no deja huella en el historial). El resultado es un historial perfectamente ordenado aunque el trabajo real fue interrumpido.
</details>

---
### Ejercicio 5 (reto) — Script de limpieza de historial

**Enunciado:** Escribe una función Python `analizar_historial(ruta_repo)` que use `subprocess` para:

1. Detectar commits con mensajes problemáticos: muy cortos (< 10 chars), sin prefijo `tipo:`, o que contienen palabras como `"wip"`, `"temp"`, `"fix"`, `"asdf"`.
2. Detectar commits muy grandes (> 5 archivos modificados).
3. Mostrar un informe con sugerencias: qué commits mejorar y cómo.

No hace ningún cambio — solo analiza e informa.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
import subprocess


<details>
<summary>💡 Ver solución — Ejercicio 5</summary>

```python
import subprocess, re
from pathlib import Path

PREFIJOS_VALIDOS = {"feat", "fix", "docs", "refactor", "test", "chore", "init", "style", "perf"}
PALABRAS_WIP     = {"wip", "temp", "tmp", "asdf", "xxx", "todo", "borrador"}

def git_cmd(args, cwd):
    r = subprocess.run(["git"] + args, cwd=cwd, capture_output=True, text=True)
    return r.stdout.strip()

def analizar_historial(ruta_repo):
    ruta = str(Path(ruta_repo).expanduser().resolve())

    # Obtener todos los commits: hash|mensaje
    log = git_cmd(["log", "--pretty=format:%h|%s"], ruta)
    if not log:
        print("Repositorio vacío.")
        return

    problemas = []

    for linea in log.splitlines():
        hash_, _, mensaje = linea.partition("|")
        issues = []

        # Mensaje muy corto
        if len(mensaje) < 10:
            issues.append(f"mensaje muy corto ({len(mensaje)} chars)")

        # Sin prefijo convencional
        tiene_prefijo = any(mensaje.lower().startswith(p + ":") for p in PREFIJOS_VALIDOS)
        if not tiene_prefijo:
            issues.append("sin prefijo convencional (feat:, fix:, docs:…)")

        # Palabras de trabajo en progreso
        palabras = set(re.findall(r'\w+', mensaje.lower()))
        wip_encontradas = palabras & PALABRAS_WIP
        if wip_encontradas:
            issues.append(f"contiene '{', '.join(wip_encontradas)}'")

        # Commit con muchos archivos
        stat = git_cmd(["show", "--stat", "--format=", hash_], ruta)
        archivos_cambiados = len([l for l in stat.splitlines() if "|") in l or " Bin" in l])
        if archivos_cambiados > 5:
            issues.append(f"demasiados archivos ({archivos_cambiados}) — considera dividirlo")

        if issues:
            problemas.append((hash_, mensaje, issues))

    # Informe
    total = len(log.splitlines())
    print(f"\n{'═'*55}")
    print(f"  ANÁLISIS DE HISTORIAL — {Path(ruta).name}")
    print(f"  {total} commits analizados | {len(problemas)} con sugerencias")
    print(f"{'═'*55}")

    if not problemas:
        print("  ✅ Historial impecable — nada que mejorar")
    else:
        for hash_, mensaje, issues in problemas:
            print(f"\n  {hash_}  \"{mensaje}\"")
            for issue in issues:
                print(f"    ⚠️  {issue}")

    print(f"\n{'═'*55}")

analizar_historial("~/git_avanzado_demo")
```
</details>

---
## Resumen de la clase

```bash
# ─── Ver cambios ───────────────────────────────────────────
git diff                        # working vs stage
git diff --staged               # stage vs último commit
git diff <hash1> <hash2>        # entre dos commits
git blame <archivo>             # quién escribió cada línea

# ─── Deshacer (de menos a más destructivo) ─────────────────
git restore --staged <archivo>  # sacar del stage
git restore <archivo>           # descartar cambios en disco
git revert <hash>               # deshacer commit (SEGURO)
git reset --soft  HEAD~n        # mover HEAD, guardar stage y disco
git reset --mixed HEAD~n        # mover HEAD, guardar solo disco
git reset --hard  HEAD~n        # borrar TODO ⚠️

# ─── Stash ─────────────────────────────────────────────────
git stash push -m "descripción" # guardar trabajo a medias
git stash list                  # ver stashes
git stash pop                   # recuperar y eliminar
git stash apply                 # recuperar sin eliminar

# ─── Amend ─────────────────────────────────────────────────
git commit --amend -m "nuevo"   # corregir mensaje
git commit --amend --no-edit    # añadir archivo olvidado

# ─── Log avanzado ──────────────────────────────────────────
git log --oneline --graph --all
git log --grep="feat"           # filtrar por mensaje
git log -S "texto"              # commits que tocaron esa cadena
git log -p -- archivo.py        # historial de un archivo

# ─── Tags ──────────────────────────────────────────────────
git tag -a v1.0 -m "Versión 1.0"
git tag                         # listar tags
```

## ¿Qué viene en la próxima clase?

En la **Clase 3** conectamos el repositorio local con el mundo: **GitHub y repositorios remotos** — `remote`, `clone`, `push`, `pull`, `fetch` y la diferencia entre SSH y HTTPS.